1. Downlaod the data

In [ ]:
from pickletools import optimize

from torch.xpu import device

from dataset.data import download_udacity_dataset
download_udacity_dataset()

2. preprocessing

In [ ]:
import pandas as pd
#Shape
df = pd.read_csv('data/training_frames_keypoints.csv')
print(df.shape)
print(df.head(2))
print("Total Nan:",df.isnull().sum().sum())
print(df.isnull().sum()[df.isnull().sum()>0])

load the data and do preprocessing

In [1]:
from src.data_pipeline import FacialKeypointsDataset
csv_path = 'data/training_frames_keypoints.csv' #label
img_dir = 'data/training' #feature
dataset = FacialKeypointsDataset(csv_file=csv_path, root_dir=img_dir) #preprocessing
print("preprocessing done")

preprocessing done


In [2]:
from torch.utils.data import DataLoader #knowing the shape
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)
batch = next(iter(loader))
images, keypoints, masks = batch['image'], batch['keypoints'], batch['mask']
print(f"images.shape: {images.shape}, keypoints.shape: {keypoints.shape}, masks.shape: {masks.shape}")

images.shape: torch.Size([32, 1, 96, 96]), keypoints.shape: torch.Size([32, 90, 2]), masks.shape: torch.Size([32, 90])


In [3]:
#convert the photo to .pt file
import torch
all_images = []
all_keypoints = []
all_masks = []
for batch in loader:
    all_images.append(batch['image'])
    all_keypoints.append(batch['keypoints'])
    all_masks.append(batch['mask'])
all_images = torch.cat(all_images, dim=0)
all_keypoints = torch.cat(all_keypoints, dim=0)
all_masks = torch.cat(all_masks, dim=0)
torch.save(all_images, 'data/all_images.pt')
torch.save(all_keypoints, 'data/all_keypoints.pt')
torch.save(all_masks, 'data/all_masks.pt')
print("saved all_images.pt, all_keypoints.pt, all_masks.pt")

saved all_images.pt, all_keypoints.pt, all_masks.pt


Train the model

In [4]:
import torch
from models.naimishnet import NaimishNet

#Gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
images_gpu = all_images.to(device)
keypoints_gpu = all_keypoints.to(device)
masks_gpu = all_masks.to(device)

#Optimizer and loss function
model = NaimishNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss(reduce='none') #loss function

batch_size = 32
num_samples = images_gpu.size(0)

for epoch in range(10):
    model.train()
    running_loss = 0.0
    for i in range(0, num_samples, batch_size):
        inputs = images_gpu[i:i+batch_size]
        targets = keypoints_gpu[i:i+batch_size]
        masks = masks_gpu[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(inputs)
        outputs = outputs.view(-1, 90, 2) #reshape to (batch_size, 90, 2)

        loss = criterion(outputs, targets)
        masked_loss = loss * masks.unsqueeze(-1)
        mean_loss = masked_loss.sum() / masks.sum()


        mean_loss.backward()
        optimizer.step()

        running_loss += mean_loss.item()

    print(f"Epoch [{epoch+1}/10], Loss: {running_loss:.4f}")

torch.save(model.state_dict(), 'output/naimishnet.pth')
print("Model saved to output/naimishnet.pth")


/home/hinze/.pyenv/versions/3.12.7/lib/python3.12/site-packages/torch/nn/_reduction.py:51: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  warnings.warn(warning.format(ret))


Epoch [1/10], Loss: 20.7378
Epoch [2/10], Loss: 14.0704
Epoch [3/10], Loss: 13.9562
Epoch [4/10], Loss: 13.4661
Epoch [5/10], Loss: 12.9084
Epoch [6/10], Loss: 12.7687
Epoch [7/10], Loss: 11.9012
Epoch [8/10], Loss: 11.9492
Epoch [9/10], Loss: 11.8474
Epoch [10/10], Loss: 11.4013
Model saved to output/naimishnet.pth
